In [1]:
using LorentzianSimplexSolver;

In [2]:
# ------------------------------------------------------------
# 1. Precision choice (user-controlled)
# ------------------------------------------------------------
const ScalarT = Float64
#const ScalarT = BigFloat

if ScalarT === BigFloat
    LorentzianSimplexSolver.PrecisionUtils.set_big_precision!(256)
    LorentzianSimplexSolver.PrecisionUtils.set_tolerance!(sqrt(eps(BigFloat)))
else
    LorentzianSimplexSolver.PrecisionUtils.set_tolerance!(1e-10)
end

# ------------------------------------------------------------
# 2. Read simplices
# ------------------------------------------------------------
simplices = [[1, 2, 4, 8, 16], [1, 2, 4, 12, 16], [1, 2, 6, 8, 16], [1, 3, 4, 8, 16], [1, 2, 6, 14, 16], [1, 5, 6, 8, 16], [1, 3, 4, 12, 16], [1, 3, 7, 8, 16], [1, 3, 7, 15, 16], [1, 5, 7, 8, 16], [1, 5, 6, 14, 16], [1, 5, 7, 15, 16], [1, 2, 10, 12, 16], [1, 2, 10, 14, 16], [1, 9, 10, 12, 16], [1, 3, 11, 12, 16], [1, 3, 11, 15, 16], [1, 9, 11, 12, 16], [1, 9, 10, 14, 16], [1, 9, 11, 15, 16], [1, 5, 13, 14, 16], [1, 5, 13, 15, 16], [1, 9, 13, 14, 16], [1, 9, 13, 15, 16]]

ns = length(simplices)

all_vertices = unique(Iterators.flatten(simplices))
sort!(all_vertices)

Nverts = length(all_vertices)

# ------------------------------------------------------------
# 3. Read vertex coordinates
# ------------------------------------------------------------
vertex_coords = Dict{Int, Vector{ScalarT}}()    

coords_lines = [
    "0, 0, 0, 0",
    "0, 0, 0, 1",
    "0, 0, 1, 0",
    "0, 0, 1, 1",
    "0, 1, 0, 0",
    "0, 1, 0, 1",
    "0, 1, 1, 0",
    "0, 1, 1, 1",
    "1//2, 0, 0, 0",
    "1//2, 0, 0, 1",
    "1//2, 0, 1, 0",
    "1//2, 0, 1, 1",
    "1//2, 1, 0, 0",
    "1//2, 1, 0, 1",
    "1//2, 1, 1, 0",
    "1//2, 1, 1, 1",
]


for (v, line) in zip(all_vertices, coords_lines)
    vertex_coords[v] = LorentzianSimplexSolver.PrecisionUtils.parse_numeric_line(line, ScalarT)
end

In [3]:
# ------------------------------------------------------------
# 4. Build geometry
# ------------------------------------------------------------
datasets = LorentzianSimplexSolver.GeometryTypes.GeometryDataset{ScalarT}[]

for (s, simplex) in enumerate(simplices)
    println("\n--- Processing simplex $s with vertices $simplex ---")
    bdypoints = [vertex_coords[v] for v in simplex]
    ds = LorentzianSimplexSolver.GeometryPipeline.run_geometry_pipeline(bdypoints)
    push!(datasets, ds)
end

geom_base = LorentzianSimplexSolver.GeometryTypes.GeometryCollection(datasets);


--- Processing simplex 1 with vertices [1, 2, 4, 8, 16] ---

--- Processing simplex 2 with vertices [1, 2, 4, 12, 16] ---

--- Processing simplex 3 with vertices [1, 2, 6, 8, 16] ---

--- Processing simplex 4 with vertices [1, 3, 4, 8, 16] ---

--- Processing simplex 5 with vertices [1, 2, 6, 14, 16] ---

--- Processing simplex 6 with vertices [1, 5, 6, 8, 16] ---

--- Processing simplex 7 with vertices [1, 3, 4, 12, 16] ---

--- Processing simplex 8 with vertices [1, 3, 7, 8, 16] ---

--- Processing simplex 9 with vertices [1, 3, 7, 15, 16] ---

--- Processing simplex 10 with vertices [1, 5, 7, 8, 16] ---

--- Processing simplex 11 with vertices [1, 5, 6, 14, 16] ---

--- Processing simplex 12 with vertices [1, 5, 7, 15, 16] ---

--- Processing simplex 13 with vertices [1, 2, 10, 12, 16] ---

--- Processing simplex 14 with vertices [1, 2, 10, 14, 16] ---

--- Processing simplex 15 with vertices [1, 9, 10, 12, 16] ---

--- Processing simplex 16 with vertices [1, 3, 11, 12, 16] ---

--

In [4]:
# ------------------------------------------------------------
# 6. Connect simplices + face matching + gauge fixing
# ------------------------------------------------------------
if ns > 1
    # println("\nFixing global κ-sign orientation ...")
    LorentzianSimplexSolver.KappaOrientation.fix_kappa_signs!(simplices, geom_base)

    # println("\nBuilding global connectivity ...")
    conn = LorentzianSimplexSolver.FourSimplexConnectivity.build_global_connectivity(simplices, geom_base)
    push!(geom_base.connectivity, conn)

    geom_ref    = deepcopy(geom_base)
    geom_parity = deepcopy(geom_base)
    LorentzianSimplexSolver.FaceXiMatching.run_face_xi_matching(geom_ref; sector=:ref)
    LorentzianSimplexSolver.FaceXiMatching.run_face_xi_matching(geom_parity; sector=:parity)
    # println("Global connectivity constructed for both relevant orientations.")

    # println("\nPerform SU(2) and SU(1,1) gauge fixing ...")
    LorentzianSimplexSolver.GaugeFixingSU.run_su2_su11_gauge_fix(geom_ref)
    LorentzianSimplexSolver.GaugeFixingSU.run_su2_su11_gauge_fix(geom_parity)
    # println("\nGauge fixing finished.")
end

  SL(2,C) matrices updated:      ✓
  SL(2,C) parity matrices updated: ✓
  Boundary bivectors updated:    ✓
  boundary ξ variables updated:  ✓
  SU(2)/SU(1,1) elements updated: ✓
  SO(1,3) frames corrected:       ✓

  SL(2,C) matrices updated:      ✓
  SL(2,C) parity matrices updated: ✓
  Boundary bivectors updated:    ✓
  boundary ξ variables updated:  ✓
  SU(2)/SU(1,1) elements updated: ✓
  SO(1,3) frames corrected:       ✓



In [5]:
# ------------------------------------------------------------
# 7a. Symbols and action (reference orientation)
# ------------------------------------------------------------
LorentzianSimplexSolver.DefineSymbols.run_define_variables(geom_ref)

sd_ref, _ = LorentzianSimplexSolver.SolveVars.run_solver(geom_ref)

S_ref = LorentzianSimplexSolver.DefineAction.compute_action(geom_ref)

S_ref_fn, labels_ref =
    LorentzianSimplexSolver.SymbolicToJulia.build_action_function(S_ref, sd_ref)

using Symbolics
@variables γ
args_ref =
    LorentzianSimplexSolver.SymbolicToJulia.build_argument_vector(sd_ref, labels_ref, γ)
args_ref_keep_j =
    LorentzianSimplexSolver.SymbolicToJulia.build_argument_vector_keep_j(sd_ref, labels_ref, γ)

S_ref_sym = expand(simplify(S_ref_fn(args_ref...)))
S_ref_sym_keep_j = expand(simplify(S_ref_fn(args_ref_keep_j...)));

In [6]:
# ------------------------------------------------------------
# 7b. Symbols and action (parity orientation)
# ------------------------------------------------------------
LorentzianSimplexSolver.DefineSymbols.run_define_variables(geom_parity)

sd_parity, _ = LorentzianSimplexSolver.SolveVars.run_solver(geom_parity)

S_parity = LorentzianSimplexSolver.DefineAction.compute_action(geom_parity)

S_parity_fn, labels_parity =
    LorentzianSimplexSolver.SymbolicToJulia.build_action_function(S_parity, sd_parity)

args_parity =
    LorentzianSimplexSolver.SymbolicToJulia.build_argument_vector(sd_parity, labels_parity, γ)
args_parity_keep_j =
    LorentzianSimplexSolver.SymbolicToJulia.build_argument_vector_keep_j(sd_parity, labels_parity, γ)

S_parity_sym = expand(simplify(S_parity_fn(args_parity...)))
S_parity_sym_keep_j = expand(simplify(S_ref_fn(args_parity_keep_j...)));

In [9]:
S_regge_num,  S_regge_symbolics = LorentzianSimplexSolver.ReggeAction.run_Regge_action(geom_ref, γ);

In [14]:
orientation = LorentzianSimplexSolver.OrientationSelector.select_orientation(S_ref_sym_keep_j, S_parity_sym_keep_j, S_regge_symbolics, γ)

phase = (S_ref_sym + S_parity_sym)/2

if orientation == :ref_neg || orientation == :parity_pos
    S_pos = expand(simplify(S_parity_sym - phase))
    S_neg = expand(simplify(S_ref_sym - phase))
else
    S_pos = expand(simplify(S_ref_sym - phase))
    S_neg = expand(simplify(S_parity_sym - phase))
end

println("Action at the positive-orientation critical point: $S_pos")
println("Action at the negative-orientation critical point: $S_neg")

Action at the positive-orientation critical point: -0.9759677134264283im
Action at the negative-orientation critical point: 0.9759677134264283im
